# Tuần 18: Pruning & Random Forest
## Nâng cao Cây Quyết Định — Chữa bệnh "Học Vẹt" và Sức Mạnh Đám Đông

---

### Bài toán hôm nay giải quyết gì?

Tuần 17 ta đã học **Cây Quyết Định (Decision Tree)** — một mô hình rất trực quan. Nhưng nó có một **vấn đề lớn**: khi cây mọc quá cao, nó **"học vẹt"** dữ liệu huấn luyện — nhớ từng chi tiết nhỏ nhặt thay vì học quy luật tổng quát.

**Ví dụ thực tế:**
> Giống như một học sinh học tủ đề thi — thuộc lòng từng bài mẫu nhưng khi thi gặp đề mới thì... bí!

Tuần này ta học **2 cách giải quyết**:

| Kỹ thuật | Ý tưởng | Analogy |
|----------|---------|----------|
| **Pruning (Tỉa cây)** | Cắt bớt những nhánh không cần thiết | Cắt tỉa cây cảnh cho gọn đẹp |
| **Random Forest (Rừng ngẫu nhiên)** | Dùng nhiều cây, lấy ý kiến đa số | Hỏi ý kiến nhiều chuyên gia |

**Dữ liệu:** Khảo sát sự hài lòng của ~127,000 hành khách (thực tế từ hãng hàng không/dịch vụ).

In [ ]:
# ============================================================
# IMPORT THƯ VIỆN
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Cấu hình biểu đồ đẹp hơn
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

print('✅ Import xong!')

---
## Phần 1: Chuẩn bị dữ liệu

Dữ liệu gồm thông tin hành khách và các đánh giá dịch vụ (thang 1–5), mục tiêu dự đoán: **khách có hài lòng không?**

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('data_decision_tree_bt.csv')
print(f'Kích thước dữ liệu: {df.shape}')
print(f'\nCột trong dữ liệu:\n{list(df.columns)}')
df.head(3)

In [ ]:
# Xem phân bố nhãn mục tiêu
counts = df['DoHaiLong'].value_counts()
print('Phân bố nhãn DoHaiLong:')
print(counts)

counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black')
plt.title('Số lượng khách Hài Lòng vs Không Hài Lòng')
plt.xlabel('Mức độ hài lòng')
plt.ylabel('Số lượng')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Tiền xử lý: chuyển cột chữ thành số
# Vì Machine Learning chỉ hiểu số, không hiểu chữ
le = LabelEncoder()

df['LoaiKhachHang_enc'] = le.fit_transform(df['LoaiKhachHang'])
df['LoaiHinh_enc'] = le.fit_transform(df['LoaiHinh'])
df['DoHaiLong_enc'] = le.fit_transform(df['DoHaiLong'])

# Chọn feature (X) và nhãn (Y)
feature_cols = ['LoaiKhachHang_enc', 'Tuoi', 'LoaiHinh_enc', 'DoTrePhut',
                'GheNgoi', 'ThucAn', 'GateLocation', 'DichVuGuiXe',
                'BauKhongKhiTranDau', 'ChatLuongTranDau', 'TraiNghiemDatVeOnline',
                'ChatLuongCacDichVuDiKem', 'DoAnToan', 'ThoiTiet',
                'SuSachSe', 'FanDoiNha', 'ThietKeLoiDi']

X = df[feature_cols]
Y = df['DoHaiLong_enc']

# Chia train/test (80% train, 20% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} mẫu')
print(f'Test:  {X_test.shape[0]} mẫu')

---
## Phần 2: Vấn đề Overfitting — "Học Vẹt"

### Overfitting là gì?

Khi cây quyết định **mọc không giới hạn**, nó sẽ:
- **Nhớ thuộc lòng** tất cả dữ liệu huấn luyện → accuracy train rất cao (~100%)
- **Không tổng quát hóa** được → accuracy test thấp hơn nhiều

```
     Train accuracy    Test accuracy
Cây đầy đủ:  99.96%   →   89.07%      ← Chênh lệch lớn = Overfitting!
```

**Analogy:** Như học sinh thuộc lòng đáp án, nhưng không hiểu bài. Đổi đề là sai ngay.

### Cách nhận biết Overfitting:

| Dấu hiệu | Ý nghĩa |
|----------|----------|
| Train accuracy >> Test accuracy | Mô hình học vẹt |
| Cây có độ sâu rất lớn (depth > 20) | Cây quá phức tạp |
| Số lá quá nhiều (leaves > 1000) | Quá chi tiết |


In [ ]:
# Huấn luyện cây quyết định KHÔNG giới hạn độ sâu
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, Y_train)

# Tính accuracy
acc_train = accuracy_score(Y_train, tree_full.predict(X_train))
acc_test  = accuracy_score(Y_test,  tree_full.predict(X_test))

print('=== Cây KHÔNG giới hạn (Full Tree) ===')
print(f'Độ sâu cây:         {tree_full.get_depth()}')
print(f'Số lá:              {tree_full.get_n_leaves()}')
print(f'Train accuracy:     {acc_train*100:.2f}%')
print(f'Test accuracy:      {acc_test*100:.2f}%')
print(f'\n⚠️  Chênh lệch: {(acc_train - acc_test)*100:.2f}% → DẤU HIỆU OVERFITTING!')

---
## Phần 3: Pruning — Tỉa cây để chữa Overfitting

### Pruning là gì?

**Pruning** (tỉa cây) = cắt bớt những nhánh không thực sự cần thiết.

**Analogy:** Như cắt tỉa cây cảnh — bỏ những cành rườm rà để cây khỏe mạnh và đẹp hơn.

### Cost Complexity Pruning (Tỉa theo độ phức tạp)

Sklearn dùng kỹ thuật **Cost Complexity Pruning** với tham số **`ccp_alpha`** (alpha):

$$\text{Chi phí} = \text{Lỗi} + \alpha \times \text{Số lá}$$

- **`alpha = 0`**: giữ nguyên cây (không tỉa)
- **`alpha lớn`**: tỉa mạnh → cây nhỏ hơn, đơn giản hơn
- **Mục tiêu**: tìm `alpha` tốt nhất để **test accuracy cao nhất**

> **Giải thích đơn giản:** alpha = "phí phạt" cho mỗi lá thêm vào. Alpha càng lớn, cây càng muốn dùng ít lá.

In [ ]:
# Bước 1: Tìm tất cả các giá trị alpha có thể
# cost_complexity_pruning_path() trả về danh sách alpha và impurity tương ứng
path = tree_full.cost_complexity_pruning_path(X_train, Y_train)
ccp_alphas = path.ccp_alphas

print(f'Tổng số alpha có thể: {len(ccp_alphas)}')
print(f'Alpha nhỏ nhất: {ccp_alphas[0]:.6f}')
print(f'Alpha lớn nhất: {ccp_alphas[-1]:.4f}')

In [ ]:
# Bước 2: Lấy mẫu 40 alpha để thử nghiệm (không cần thử tất cả)
sample_indices = np.linspace(0, len(ccp_alphas) - 1, 40, dtype=int)
sample_alphas = ccp_alphas[sample_indices]

# Bước 3: Với mỗi alpha, huấn luyện cây và ghi lại accuracy
train_accs = []
test_accs  = []

for alpha in sample_alphas:
    clf = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf.fit(X_train, Y_train)
    train_accs.append(accuracy_score(Y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(Y_test, clf.predict(X_test)))

# Vẽ biểu đồ: alpha vs accuracy
plt.figure(figsize=(11, 5))
plt.plot(sample_alphas, train_accs, 'b-o', markersize=4, label='Train accuracy')
plt.plot(sample_alphas, test_accs,  'r-o', markersize=4, label='Test accuracy')

# Đánh dấu điểm tốt nhất
best_idx   = np.argmax(test_accs)
best_alpha = sample_alphas[best_idx]
best_acc   = test_accs[best_idx]
plt.axvline(best_alpha, color='green', linestyle='--', label=f'Best alpha={best_alpha:.6f}')
plt.scatter([best_alpha], [best_acc], color='green', s=150, zorder=5)

plt.title('Pruning: Accuracy theo giá trị alpha')
plt.xlabel('ccp_alpha (độ tỉa cây)')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\n✅ Alpha tốt nhất: {best_alpha:.6f}')
print(f'✅ Test accuracy tốt nhất: {best_acc*100:.2f}%')

In [ ]:
# Bước 4: Huấn luyện cây được tỉa với alpha tốt nhất
tree_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
tree_pruned.fit(X_train, Y_train)

pruned_train = accuracy_score(Y_train, tree_pruned.predict(X_train))
pruned_test  = accuracy_score(Y_test,  tree_pruned.predict(X_test))

print('=== SO SÁNH: Cây đầy đủ vs Cây được tỉa ===')
print(f'{"":30} {"Full Tree":>12} {"Pruned Tree":>12}')
print('-' * 56)
print(f'{"Độ sâu cây":30} {tree_full.get_depth():>12} {tree_pruned.get_depth():>12}')
print(f'{"Số lá":30} {tree_full.get_n_leaves():>12} {tree_pruned.get_n_leaves():>12}')
print(f'{"Train accuracy":30} {acc_train*100:>11.2f}% {pruned_train*100:>11.2f}%')
print(f'{"Test accuracy":30} {acc_test*100:>11.2f}% {pruned_test*100:>11.2f}%')
print(f'{"Chênh lệch train-test":30} {(acc_train-acc_test)*100:>11.2f}% {(pruned_train-pruned_test)*100:>11.2f}%')
print(f'\n✅ Sau Pruning: test accuracy tăng {(pruned_test - acc_test)*100:.2f}%, cây gọn hơn nhiều!')

---
## Phần 4: Random Forest — Sức Mạnh Đám Đông

### Ý tưởng chính

**Random Forest** = Xây nhiều cây quyết định **khác nhau**, rồi **bỏ phiếu đa số**.

**Analogy 1:** Thay vì hỏi 1 bác sĩ → hỏi **100 bác sĩ** → lấy kết luận đa số!

**Analogy 2:** Như cuộc thi gameshow "Ai là triệu phú" — khi khó, dùng "hỏi khán giả" vì đám đông thường đúng hơn một người.

### Cách Random Forest tạo sự đa dạng

| Kỹ thuật | Giải thích |
|----------|------------|
| **Bootstrap sampling** | Mỗi cây học trên tập con **ngẫu nhiên** của dữ liệu |
| **Random feature selection** | Mỗi lần chia nhánh, chỉ xem xét **một số feature ngẫu nhiên** |
| **Bỏ phiếu đa số** | Kết quả cuối = nhãn được **nhiều cây nhất chọn** |

### Tham số quan trọng

| Tham số | Ý nghĩa | Gợi ý |
|---------|---------|-------|
| `n_estimators` | Số cây trong rừng | 100–200 thường đủ tốt |
| `max_depth` | Độ sâu tối đa mỗi cây | Hạn chế để tránh overfitting |
| `random_state` | Hạt giống ngẫu nhiên | Để kết quả tái lặp được |

### OOB Score — "Điểm kiểm tra miễn phí"

Vì mỗi cây chỉ học trên ~63% dữ liệu (bootstrap), 37% còn lại gọi là **Out-Of-Bag (OOB)**. Ta dùng phần này để **tự đánh giá** mà không cần tập test riêng!


In [ ]:
# Huấn luyện Random Forest cơ bản
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
rf.fit(X_train, Y_train)

rf_train = accuracy_score(Y_train, rf.predict(X_train))
rf_test  = accuracy_score(Y_test,  rf.predict(X_test))

print('=== Random Forest (100 cây) ===')
print(f'Train accuracy: {rf_train*100:.2f}%')
print(f'Test accuracy:  {rf_test*100:.2f}%')
print(f'OOB score:      {rf.oob_score_*100:.2f}%  ← đánh giá tự động, không cần tập test!')
print(f'\n✅ So sánh với cây đơn đã tỉa: test accuracy {pruned_test*100:.2f}% → {rf_test*100:.2f}%')

In [ ]:
# Thử nghiệm: số cây ảnh hưởng đến độ chính xác như thế nào?
n_estimators_list = [10, 20, 30, 50, 75, 100, 150, 200]
oob_errors = []
test_accs_rf = []

for n in n_estimators_list:
    rf_temp = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, Y_train)
    oob_errors.append(1 - rf_temp.oob_score_)          # OOB error = 1 - OOB accuracy
    test_accs_rf.append(accuracy_score(Y_test, rf_temp.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: OOB Error
axes[0].plot(n_estimators_list, oob_errors, 'b-o', markersize=6)
axes[0].set_title('OOB Error theo số cây')
axes[0].set_xlabel('Số cây (n_estimators)')
axes[0].set_ylabel('OOB Error (càng nhỏ càng tốt)')

# Biểu đồ 2: Test Accuracy
axes[1].plot(n_estimators_list, [a*100 for a in test_accs_rf], 'r-o', markersize=6)
axes[1].set_title('Test Accuracy theo số cây')
axes[1].set_xlabel('Số cây (n_estimators)')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_ylim([89, 94])

plt.tight_layout()
plt.show()

print('Kết quả theo số cây:')
for n, oob, tacc in zip(n_estimators_list, oob_errors, test_accs_rf):
    print(f'  {n:>3} cây: OOB Error={oob*100:.2f}%, Test Acc={tacc*100:.2f}%')

---
## Phần 5: Feature Importance — Feature nào quan trọng nhất?

Random Forest không chỉ dự đoán tốt mà còn cho ta biết **feature (đặc trưng) nào đóng góp nhiều nhất** vào kết quả.

**Ý nghĩa thực tế:** Biết được yếu tố nào ảnh hưởng nhất đến sự hài lòng của khách hàng → tập trung cải thiện đúng chỗ!

Feature importance trong sklearn = **mức độ giảm impurity trung bình** khi feature đó được dùng để chia nhánh.

In [ ]:
# Lấy feature importance từ mô hình Random Forest
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)  # sắp xếp tăng dần để barh đẹp

# Vẽ biểu đồ ngang
plt.figure(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_df)))
plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors, edgecolor='grey')
plt.title('Feature Importance — Yếu tố nào quan trọng nhất?', fontsize=14)
plt.xlabel('Mức độ quan trọng')
plt.tight_layout()
plt.show()

print('\nTop 5 feature quan trọng nhất:')
top5 = feat_df.sort_values('Importance', ascending=False).head(5)
for i, (_, row) in enumerate(top5.iterrows(), 1):
    print(f'  {i}. {row["Feature"]:30s}: {row["Importance"]*100:.2f}%')

---
## Phần 6: So sánh tổng thể

Bây giờ ta so sánh **3 mô hình** đã học trong 2 tuần:

In [ ]:
# Tổng hợp so sánh các mô hình
models = ['Decision Tree\n(Full)', 'Decision Tree\n(Pruned)', 'Random Forest\n(100 cây)']
train_scores = [acc_train * 100, pruned_train * 100, rf_train * 100]
test_scores  = [acc_test  * 100, pruned_test  * 100, rf_test  * 100]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, train_scores, width, label='Train accuracy', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores,  width, label='Test accuracy',  color='coral',     alpha=0.8)

# Thêm nhãn số lên từng cột
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_title('So sánh Train vs Test Accuracy — 3 mô hình', fontsize=14)
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim([85, 102])
ax.legend()
ax.axhline(100, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('Nhận xét:')
print('  - Full Tree: Train ~100%, Test thấp hơn nhiều → OVERFITTING')
print('  - Pruned Tree: Cân bằng hơn, test accuracy tốt hơn')
print('  - Random Forest: Test accuracy cao nhất → MÔ HÌNH TỐT NHẤT!')

---
## Tóm tắt & Ghi nhớ

### 7 điểm chính cần nhớ

| # | Điểm chính | Chi tiết |
|---|-----------|----------|
| 1 | **Overfitting** | Cây quá sâu → học vẹt → train cao, test thấp |
| 2 | **Pruning** | Tỉa bớt nhánh không cần thiết → giảm overfitting |
| 3 | **ccp_alpha** | Tham số kiểm soát mức độ tỉa (alpha lớn → tỉa nhiều) |
| 4 | **Random Forest** | Nhiều cây khác nhau → bỏ phiếu đa số → chính xác hơn |
| 5 | **Bootstrap** | Mỗi cây học trên tập con ngẫu nhiên → tạo đa dạng |
| 6 | **OOB Score** | Tự đánh giá mô hình không cần tập test riêng |
| 7 | **Feature Importance** | Random Forest cho biết feature nào quan trọng nhất |

### So sánh với tuần trước (Week 17)

```
Week 17: Decision Tree cơ bản
    ↓ Vấn đề: Overfitting
Week 18: Giải pháp 1 → Pruning (tỉa cây)
         Giải pháp 2 → Random Forest (nhiều cây)
```

### Khi nào dùng gì?

| Tình huống | Nên dùng |
|------------|----------|
| Cần giải thích được kết quả | Decision Tree (có Pruning) |
| Cần độ chính xác cao nhất | Random Forest |
| Muốn biết feature quan trọng | Random Forest (feature importance) |
| Dữ liệu ít, cần nhanh | Decision Tree |

---
> **Bước tiếp theo:** Tuần 19 — K-Means Clustering (phân cụm dữ liệu không có nhãn!)